# Kernel

### Design
- We spawn as many CTAs as we have C-tiles (bM, bN)
- Each CTA will loop-through all K-tiles (= K/bK) or 8K/64 if K = 64 i.e 128 tiles
- GMEM Latency is hidden by CTA-parallelism on same SM
   - Here, numCTiles = (M/bM * N/bN) = (8K * 8K/(128*256)) = 2048
   - Number of CTAs per 128 SMs = 2048, CTAs per SM = 16

### Is this enough CTAs to hide GMEM latency?
- HBM BW per device = 8 TBps or 8192/128 GBps per SM
- 64 GBps per SM
- Freq ~= 2 GHz or 2e9
- GMEM Latency ~= 1000 cycles = 1e3 * 0.5e-9 = 5e-7 secs
- Data inflight to hide latency = 64e9 bytes/secs * 5e-7 secs
- Data inflight reqd = 34 KB
- Each CTA brings in bMxbK * bNxbK tiles each iter = (128x64 + 256x64) x2B (bf16)
- Each CTA brings in 48 KB, hence a second CTA is enough to hide GMEM latency

### ILP in K-tiles
- Each K-tile can be software pipelined in two ways:
- Prefetch N K-tiles to hide GMEM latency further
- Accumulate 2-4 K-tiles unrolled together to increase ILP with limited CTAs
- This will increase register/SMEM pressure
- This is done by Triton intrinsically when lowering tl.dot()

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def gemm_kernel_rowMajorBlocking(
    A: torch.tensor, B: torch.tensor, C: torch.tensor,
    M: tl.int32, N: tl.int32, K: tl.int32,
    blkM: tl.constexpr, blkN: tl.constexpr, blkK: tl.constexpr,
):
  ### Locate Correct block ###
  # if we match bM, bN, bK to wgmma/umma atom dimensions, tl.dot() can lower to TensorCore instructions.
  # m0 [ <---- N ------ >]
  # m1 [ <---- N ------ >]
  # m2 [ <---- N ------ >]
  # m3 [ <---- N ------ >]
  pid = tl.program_id(0)
  num_pids_n = tl.cdiv(N, blkN)
  pid_m = pid // num_pids_n
  pid_n = pid % num_pids_n

  ### Pointers/Tensors ###
  a_ptr = A.data()
  b_ptr = B.data()
  c_ptr = C.data()
  A_stride_m = A.stride(0)
  A_stride_k = A.stride(1)
  B_stride_k = B.stride(0)
  B_stride_n = B.stride(1)
  C_stride_m = C.stride(0)
  C_stride_n = C.stride(1)
  M = A.shape(0)
  N = B.shape(1)
  K = B.shape(0)

  ### Block offsets ###
  # a-blk (dim0 offsets for MxK tensor)
  offsets_am = pid_m * blkM + tl.arange(0, blkM) % blkM
  # b-blk (dim1 offsets for KxM tensor)
  offsets_bn = pid_n * blkN + tl.arange(0, blkN) % blkN
  # c-blk dim0 offsets
  offsets_cm = pid_m * blkM + tl.arange(0, blkM) # masked later
  # c-blk dim1 offsets
  offsets_cn = pid_n * blkN + tl.arange(0, blkN) # masked later
  offsets_k = tl.arange(0, blkK)
  # blkM x blkK pointers for A-blk
  a_ptrs = a_ptr + offsets_am[:,None] * A_stride_m + offsets_k[None,:] * A_stride_k
  # blkK x blkN pointers for B-blk
  b_ptrs = b_ptr + offsets_k[:,None] * B_stride_k + offsets_bn[None,:] * B_stride_n
  # blkM x blkN pointers for C-blk (stationary)
  c_ptrs = c_ptr + offsets_cm[:,None] * C_stride_m + offsets_cn[None,:] * C_stride_n
  
  ### K-tile loop ###
  k = 0
  c_blk = tl.zeros((blkM, blkN), dtype=tl.float32)
  while (k < K):
    # Important to note that a_ptrs is a 2D tensor of addresses
    # Bounds checking is best doing using offsets!!
    mask = k + offsets_k[None,:] < K
    a_blk = tl.load(a_ptrs, mask=mask, other=0.0)
    mask = k + offsets_k[:,None] < K
    b_blk = tl.load(b_ptrs, mask=mask, other=0.0)
    c_blk = tl.dot(a_blk, b_blk, c_blk)
    a_ptrs += blkK * A_stride_k # 2D broadcast to all addresses
    b_ptrs += blkK * B_stride_k # 2D broadcast to all addresses
    k += blkK
  mask = (c_ptrs[:, None] < M) & (c_ptrs[None,:] < N)
  tl.store(c_ptrs, c_blk, mask=mask)



